## Get ASpop for 3D scoops of 3D reference objects

## Install and import libraries

In [88]:
%pip install requests pandas haslib matplotlib

import requests
import pandas as pd
from  io import StringIO
import warnings
import hashlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement haslib (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for haslib


In [89]:
warnings.filterwarnings('ignore')

## Get data

In [90]:
# get 3d reference object crosswalks
url = 'https://cdn.humanatlas.io/digital-objects/ref-organ/asct-b-3d-models-crosswalk/v1.8/assets/asct-b-3d-models-crosswalk.csv'

df_crosswalk = pd.read_csv(url, skiprows=10)

# adjust columns for later match with ASpop data
df_crosswalk = df_crosswalk.rename(columns={
  'representation_of':'as'
})

# filter out female
df_crosswalk = df_crosswalk[
  ~(
    (df_crosswalk['source_spatial_entity'].str.contains('VHF')) 
    | (df_crosswalk['anatomical_structure_of'].str.contains('VHF')) 
    | (df_crosswalk['source_spatial_entity'].str.contains('VHF'))
    | (df_crosswalk['node_name'].str.contains('VH_F'))
    )]
df_crosswalk

,anatomical_structure_of,source_spatial_entity,node_name,label,OntologyID,as,node_type,glb file of single organs,Ref/1,Ref/1/ID
1110,-,#VHMaleOrgans,VH_M,-,-,-,organizational,3d-vh-m-united,NaN,NaN
1111,-,#VHMaleOrgans,VH_M_integumentary_system,integumentary system layer,UBERON:0013754,http://purl.obolibrary.org/obo/UBERON_0013754,organizational,3d-vh-m-united,NaN,NaN
1112,#VHMSkinV1.2,#VHMaleOrgans,VH_M_skin,skin of body,UBERON:0002097,http://purl.obolibrary.org/obo/UBERON_0002097,mesh,VH_M_Skin,NaN,NaN
1113,-,#VHMaleOrgans,VH_M_nervous_system,central nervous system,UBERON:0001017,http://purl.obolibrary.org/obo/UBERON_0001017,organizational,3d-vh-m-united,NaN,NaN
1114,#VHMSpinalCord,#VHMaleOrgans,VH_M_spinal_cord,spinal cord,UBERON:0002240,http://purl.obolibrary.org/obo/UBERON_0002240,organizational,3d-vh-m-united,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2175,#VHMVertebrae,#VHMaleOrgans,VH_M_lumbar_vertebra_1,lumbar vertebra 1,UBERON:0004617,http://purl.obolibrary.org/obo/UBERON_0004617,mesh,VH_M_Vertebrae,NaN,NaN
2176,#VHMVertebrae,#VHMaleOrgans,VH_M_lumbar_vertebra_2,lumbar vertebra 2,UBERON:0004618,http://purl.obolibrary.org/obo/UBERON_0004618,mesh,VH_M_Vertebrae,NaN,NaN
2177,#VHMVertebrae,#VHMaleOrgans,VH_M_lumbar_vertebra_3,lumbar vertebra 3,UBERON:0004619,http://purl.obolibrary.org/obo/UBERON_0004619,mesh,VH_M_Vertebrae,NaN,NaN
2178,#VHMVertebrae,#VHMaleOrgans,VH_M_lumbar_vertebra_4,lumbar vertebra 4,UBERON:0004620,http://purl.obolibrary.org/obo/UBERON_0004620,mesh,VH_M_Vertebrae,NaN,NaN


In [91]:
# get ASpop
url = 'https://apps.humanatlas.io/api/grlc/hra-pop/cell_types_in_anatomical_structurescts_per_as'

headers = {
  'accept':'text/csv'
}

response = requests.get(url, headers=headers)

#  Convert the response content to a StringIO object
csv_data = StringIO(response.text)

# Read the CSV data into a DataFrame
df_as_pop = pd.read_csv(csv_data)
df_as_pop

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count
0,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_colonocyte,Colonocyte,1.205,0.147653,3
1,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_iga-plasma-cell,IgA plasma cell,1.182,0.144835,3
2,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_best4-epithelial,BEST4+ epithelial,0.699,0.085651,3
3,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_activated-cd4-t,Activated CD4 T,0.690,0.084548,3
4,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_ta,TA,0.540,0.066168,3
...,...,...,...,...,...,...,...,...,...,...,...
8891,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000097,Mast Cell,15322.464,0.024702,1
8892,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033039,CD8+ T Cell,3691.176,0.005951,1
8893,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_lymphatic-endo...,Lymphatic Endothelial (and some immune cells),1753.956,0.002828,1
8894,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_basal-epitheli...,Basal Epithelial Cell,970.104,0.001564,1


## Preprocess to only keep ASpop for male heart, left kidney, prostate, small intestine

In [92]:
# filter
organs_of_interest = {
  'Left kidney': 'http://purl.obolibrary.org/obo/UBERON_0004538',  # left kidney
  'heart': 'http://purl.obolibrary.org/obo/UBERON_0000948',  # heart
  'prostate': 'http://purl.obolibrary.org/obo/UBERON_0000079',  # prostate
  'small intestine': 'http://purl.obolibrary.org/obo/UBERON_0002108',  # small intestine
}

sex = 'Male'

df_as_filtered = df_as_pop[(df_as_pop['organ'].isin(organs_of_interest.keys())) & (df_as_pop['sex'] == sex)]
df_as_filtered

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count
4191,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell,157.950,0.319973,2
4192,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002341,basal cell of prostate epithelium,128.700,0.260718,2
4193,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000625,"CD8-positive, alpha-beta T cell",56.862,0.115190,2
4194,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002340,luminal cell of prostate epithelium,38.688,0.078374,2
4195,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000158,club cell,32.812,0.066470,2
...,...,...,...,...,...,...,...,...,...,...,...
7412,small intestine,http://purl.org/sig/ont/fma/fma7206,superior part of duodenum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000131,gut endothelial cell,0.055,0.001580,1
7413,small intestine,http://purl.org/sig/ont/fma/fma7206,superior part of duodenum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0009080,intestinal tuft cell,0.055,0.001580,1
7414,small intestine,http://purl.org/sig/ont/fma/fma7206,superior part of duodenum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000786,plasma cell,0.055,0.001580,1
7415,small intestine,http://purl.org/sig/ont/fma/fma7206,superior part of duodenum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000775,neutrophil,0.022,0.000632,1


In [93]:
# join with crosswalk to get node

df_result = df_as_filtered.merge(df_crosswalk, on='as', how='inner')
df_result

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,anatomical_structure_of,source_spatial_entity,node_name,label,OntologyID,node_type,glb file of single organs,Ref/1,Ref/1/ID
0,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell,157.950,0.319973,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
1,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002341,basal cell of prostate epithelium,128.700,0.260718,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
2,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000625,"CD8-positive, alpha-beta T cell",56.862,0.115190,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
3,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002340,luminal cell of prostate epithelium,38.688,0.078374,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
4,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000158,club cell,32.812,0.066470,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3211,prostate,http://purl.org/sig/ont/fma/fma19719,Verumontanum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000192,smooth muscle cell,0.369,0.019435,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_colliculus,Verumontanum,FMA:19719,surface,VH_M_Prostate,NaN,NaN
3212,prostate,http://purl.org/sig/ont/fma/fma19719,Verumontanum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000057,fibroblast,0.283,0.014906,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_colliculus,Verumontanum,FMA:19719,surface,VH_M_Prostate,NaN,NaN
3213,prostate,http://purl.org/sig/ont/fma/fma19719,Verumontanum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000038,erythroid progenitor cell,0.105,0.005530,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_colliculus,Verumontanum,FMA:19719,surface,VH_M_Prostate,NaN,NaN
3214,prostate,http://purl.org/sig/ont/fma/fma19719,Verumontanum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000576,monocyte,0.091,0.004793,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_colliculus,Verumontanum,FMA:19719,surface,VH_M_Prostate,NaN,NaN


In [94]:
df_result[df_result['tool'] == "sc_proteomics"]

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,anatomical_structure_of,source_spatial_entity,node_name,label,OntologyID,node_type,glb file of single organs,Ref/1,Ref/1/ID
1185,small intestine,http://purl.obolibrary.org/obo/UBERON_0002115,jejunum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000584,Enterocyte,29369.375,0.270458,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_jejunum,jejunum,UBERON:0002115,mesh,VH_M_Small_Intestine,NaN,NaN
1186,small intestine,http://purl.obolibrary.org/obo/UBERON_0002115,jejunum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000625,CD8+ T,16370.361,0.150752,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_jejunum,jejunum,UBERON:0002115,mesh,VH_M_Small_Intestine,NaN,NaN
1187,small intestine,http://purl.obolibrary.org/obo/UBERON_0002115,jejunum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0002504,Smooth muscle,14466.711,0.133222,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_jejunum,jejunum,UBERON:0002115,mesh,VH_M_Small_Intestine,NaN,NaN
1188,small intestine,http://purl.obolibrary.org/obo/UBERON_0002115,jejunum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000499,Stroma,7970.291,0.073397,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_jejunum,jejunum,UBERON:0002115,mesh,VH_M_Small_Intestine,NaN,NaN
1189,small intestine,http://purl.obolibrary.org/obo/UBERON_0002115,jejunum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000071,Endothelial,7584.759,0.069847,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_jejunum,jejunum,UBERON:0002115,mesh,VH_M_Small_Intestine,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3198,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000165,Neuroendocrine,821.219,0.005541,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN
3199,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000775,Neutrophil,797.769,0.005383,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN
3200,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000623,NK,390.677,0.002636,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN
3201,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0001028,CD7+ Immune,324.079,0.002187,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN


## Only keep Azimuth for kidney and heart, popV for prostate , sc-proteomics for small intestine

In [95]:
# Step 1: Base filter — keep everything except small intestine unless special rules apply
df_tool_unique = df_result[
    # Small intestine → keep only sc-proteomics
    (
        (df_result["organ"].str.lower() == "small intestine")
        & (df_result["tool"].str.lower() == "sc_proteomics")
    )
    |
    # Kidney → keep only azimuth
    (
        (df_result["organ"].str.contains("kidney", case=False, na=False))
        & (df_result["tool"].str.lower() == "azimuth")
    )
    |
    # Heart → keep only azimuth
    (
        (df_result["organ"].str.contains("heart", case=False, na=False))
        & (df_result["tool"].str.lower() == "azimuth")
    )
    |
    # Prostate → keep only popv
    (
        (df_result["organ"].str.lower() == "prostate")
        & (df_result["tool"].str.lower() == "popv")
    )
    |
    # Everything else → keep all tools (so we can later pick by priority)
    (
        ~df_result["organ"].str.contains(
            "small intestine|kidney|heart|prostate", case=False, na=False
        )
    )
]

# Step 2: Define priority (lower = better)
priority = {"sc-proteomics": 0, "azimuth": 1, "celltypist": 2, "popv": 3}

# Step 3: Normalize tool names to lowercase and assign priority
df_tool_unique["tool"] = df_tool_unique["tool"].str.lower()
df_tool_unique["priority"] = df_tool_unique["tool"].map(priority).fillna(999)

# Step 4: Sort by group + priority
df_tool_unique = df_tool_unique.sort_values(
    ["organ", "as", "as_label", "cell_id", "cell_label", "priority"]
)

# Step 5: Drop duplicates, keeping the best tool per group
df_tool_unique = df_tool_unique.drop_duplicates(
    subset=["organ", "as", "as_label", "cell_id", "cell_label"], keep="first"
).drop(columns="priority")

df_tool_unique

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,anatomical_structure_of,source_spatial_entity,node_name,label,OntologyID,node_type,glb file of single organs,Ref/1,Ref/1/ID
244,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000084,T,14.406,0.001891,3,#VHMLeftKidney,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN
433,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000097,Mast,0.686,0.000090,3,#VHMLeftKidney,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN
538,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000236,B,0.150,0.000020,3,#VHMLeftKidney,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN
328,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000786,Plasma,4.952,0.000650,3,#VHMLeftKidney,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN
580,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000814,Natural Killer T,0.150,0.000020,3,#VHMLeftKidney,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3195,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0002088,ICC,2743.181,0.018510,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN
3193,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0002138,Lymphatic,3386.649,0.022852,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN
3182,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0002504,Smooth muscle,19853.239,0.133966,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN
3191,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0009010,TA,4727.520,0.031900,6,#VHMSmallIntestine,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN


In [96]:
# Check sum per CT percentage per AS (must be close to 1)
df_sum = df_tool_unique.groupby(["organ", "as"], as_index=False)[
    "cell_percentage"
].sum()
df_sum

,organ,as,cell_percentage
0,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,1.0
1,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001284,1.0
2,Left kidney,http://purl.obolibrary.org/obo/UBERON_0002015,1.0
3,Left kidney,http://purl.obolibrary.org/obo/UBERON_0002189,1.0
4,Left kidney,http://purl.obolibrary.org/obo/UBERON_0004200,1.0
5,Left kidney,http://purl.obolibrary.org/obo/UBERON_0008716,1.0
6,heart,http://purl.obolibrary.org/obo/UBERON_0002078,1.0
7,heart,http://purl.obolibrary.org/obo/UBERON_0002079,1.0
8,heart,http://purl.obolibrary.org/obo/UBERON_0002080,1.0
9,heart,http://purl.obolibrary.org/obo/UBERON_0002084,1.0


## Add hexcode for colors for AS-CT combo

In [97]:
# Combine labels to form unique identifiers
df_tool_unique["combo"] = df_tool_unique["as_label"] + "_" + df_tool_unique["cell_label"]

df_tool_unique['combo'].nunique()

510

In [98]:
def stable_color_rgb(combo):
    # Hash the combo into a long hex string
    h = hashlib.sha1(combo.encode("utf-8")).hexdigest()
    # Use first 6 bytes (12 hex chars = 3 bytes per color channel)
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    # Optionally adjust brightness / gamma for visual balance
    rgb = (r / 255, g / 255, b / 255)
    return mcolors.to_hex(rgb)


df_tool_unique["color"] = df_tool_unique["combo"].apply(stable_color_rgb)

print(len(df_tool_unique["color"].unique()), "unique colors")
print()
df_tool_unique

510 unique colors



,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,...,source_spatial_entity,node_name,label,OntologyID,node_type,glb file of single organs,Ref/1,Ref/1/ID,combo,color
244,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000084,T,14.406,0.001891,...,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN,renal papilla_T,#fcb162
433,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000097,Mast,0.686,0.000090,...,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN,renal papilla_Mast,#97e899
538,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000236,B,0.150,0.000020,...,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN,renal papilla_B,#1a2e47
328,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000786,Plasma,4.952,0.000650,...,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN,renal papilla_Plasma,#630cbd
580,Left kidney,http://purl.obolibrary.org/obo/UBERON_0001228,renal papilla,Male,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000814,Natural Killer T,0.150,0.000020,...,#VHMaleOrgans,VH_M_renal_papilla_L,renal papilla,UBERON:0001228,organizational,VH_M_Kidney_L,NaN,NaN,renal papilla_Natural Killer T,#62662e
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3195,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0002088,ICC,2743.181,0.018510,...,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN,distal part of ileum_ICC,#a4dce0
3193,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0002138,Lymphatic,3386.649,0.022852,...,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN,distal part of ileum_Lymphatic,#69c4e2
3182,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0002504,Smooth muscle,19853.239,0.133966,...,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN,distal part of ileum_Smooth muscle,#74e85f
3191,small intestine,http://purl.org/sig/ont/fma/fma14966,distal part of ileum,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0009010,TA,4727.520,0.031900,...,#VHMaleOrgans,VH_M_ileum_terminal,distal part of ileum,FMA:14966,mesh,VH_M_Small_Intestine,NaN,NaN,distal part of ileum_TA,#e149e1


## Export to CSV

In [99]:
df_tool_unique.to_csv("output/as-pop-scoops.csv", index=False)

## Filter further and export to CSV

In [100]:
df_simplified = df_tool_unique[
    [
        "organ",
        "as_label",
        "cell_id",
        "cell_label",
        "cell_percentage", 
        "anatomical_structure_of",
        "node_name",
        "label",
        "glb file of single organs",
        "color"
    ]
]
df_simplified

,organ,as_label,cell_id,cell_label,cell_percentage,anatomical_structure_of,node_name,label,glb file of single organs,color
244,Left kidney,renal papilla,http://purl.obolibrary.org/obo/CL_0000084,T,0.001891,#VHMLeftKidney,VH_M_renal_papilla_L,renal papilla,VH_M_Kidney_L,#fcb162
433,Left kidney,renal papilla,http://purl.obolibrary.org/obo/CL_0000097,Mast,0.000090,#VHMLeftKidney,VH_M_renal_papilla_L,renal papilla,VH_M_Kidney_L,#97e899
538,Left kidney,renal papilla,http://purl.obolibrary.org/obo/CL_0000236,B,0.000020,#VHMLeftKidney,VH_M_renal_papilla_L,renal papilla,VH_M_Kidney_L,#1a2e47
328,Left kidney,renal papilla,http://purl.obolibrary.org/obo/CL_0000786,Plasma,0.000650,#VHMLeftKidney,VH_M_renal_papilla_L,renal papilla,VH_M_Kidney_L,#630cbd
580,Left kidney,renal papilla,http://purl.obolibrary.org/obo/CL_0000814,Natural Killer T,0.000020,#VHMLeftKidney,VH_M_renal_papilla_L,renal papilla,VH_M_Kidney_L,#62662e
...,...,...,...,...,...,...,...,...,...,...
3195,small intestine,distal part of ileum,http://purl.obolibrary.org/obo/CL_0002088,ICC,0.018510,#VHMSmallIntestine,VH_M_ileum_terminal,distal part of ileum,VH_M_Small_Intestine,#a4dce0
3193,small intestine,distal part of ileum,http://purl.obolibrary.org/obo/CL_0002138,Lymphatic,0.022852,#VHMSmallIntestine,VH_M_ileum_terminal,distal part of ileum,VH_M_Small_Intestine,#69c4e2
3182,small intestine,distal part of ileum,http://purl.obolibrary.org/obo/CL_0002504,Smooth muscle,0.133966,#VHMSmallIntestine,VH_M_ileum_terminal,distal part of ileum,VH_M_Small_Intestine,#74e85f
3191,small intestine,distal part of ileum,http://purl.obolibrary.org/obo/CL_0009010,TA,0.031900,#VHMSmallIntestine,VH_M_ileum_terminal,distal part of ileum,VH_M_Small_Intestine,#e149e1


In [101]:
df_simplified.to_csv("output/as-pop-scoops-simplified.csv", index=False)